# 3b. Anonymization, part 2: boilerplate and a language model

On the text of part 1 (`articles_rules.csv.gz`), two more passes, so that "the models do not see who published the
article" is hard to attack (details and reasons in `anonymization_llm.py`):
1. **boilerplate**: a sentence repeated in at least 15% of the articles of an outlet (and in at least 3) is removed from
   that outlet (newsletter and donation calls, disclaimers, fixed sign-offs);
2. **language model**: a local instruction model (Qwen2.5-7B-Instruct, greedy) lists, article by article, the phrases that
   reveal the outlet (its name in any form, its own staff). The phrases really in the text that are not common words or
   public names (frequent in OTHER outlets) form a lexicon per outlet, replaced in ALL its articles with `[OUTLET]` /
   `[AUTHOR]`. The model never rewrites the text: every change is a deletion or an exact replacement, listed in files.
The result, `articles_anonymous.csv.gz`, is the text every later step reads. Checks: `03c_anonymization_checks`.

In [ ]:
DETECTOR_BUDGET = 24000        # tokens per batch (prompt + answer)

In [ ]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())
import json
import re

import anonymization_llm as AL

AL.WORK.mkdir(parents=True, exist_ok=True)
art = pd.read_csv(C.OUT / "articles_rules.csv.gz")               # part 1
part1 = art.text.fillna("").copy()

## Boilerplate

In [ ]:
art["text"], removed = AL.boilerplate(art)
removed.to_csv(AL.WORK / "boilerplate.csv", index=False)
w0, w1 = part1.str.split().str.len(), art.text.str.split().str.len()
print(f"sentences removed: {len(removed):,} in {removed.domain.nunique()} outlets; articles touched {(w1 < w0).mean():.1%}; "
      f"words removed {1 - w1.sum() / w0.sum():.2%}; articles under 50 words after: {(w1 < 50).sum()}")
removed.sort_values("articles", ascending=False).head(15)[["sentence", "articles", "of"]]

## The language model lists the phrases that reveal the outlet

In [ ]:
f = AL.WORK / "detector_answers.json"
answers = {int(k): v for k, v in json.load(open(f)).items()} if f.exists() else {}
if len(answers) < len(art):
    import models as MO
    MO.wait_for_gpu(18, AL.LOG)
    tok, m = AL.load_llm(AL.DETECTOR)
    save = lambda d: json.dump({str(k): v for k, v in d.items()}, open(f, "w"))
    answers = AL.generate(tok, m, AL.detection_prompts(art), 160, budget=DETECTOR_BUDGET, done=answers, save=save, logfile=AL.LOG)
    del m; import torch; torch.cuda.empty_cache()
bad = sum(1 for a in answers.values() if not re.search(r"\{.*\}", a, flags=re.S))
print(f"answers: {len(answers):,}; without JSON: {bad}")

In [ ]:
L = AL.lexicon(art, answers)
L.to_csv(AL.WORK / "lexicon.csv", index=False)
print(f"phrases proposed: {L.phrase.nunique():,} in {L.domain.nunique()} outlets; kept: {L[L.kept].groupby(['domain', 'phrase']).ngroups:,} "
      f"(outlet {L[L.kept & (L.kind == 'outlet')].phrase.nunique()}, staff {L[L.kept & (L.kind == 'staff')].phrase.nunique()})")
L[~L.kept].sort_values("elsewhere", ascending=False).drop_duplicates("phrase").head(15)[["kind", "phrase", "elsewhere"]]

In [ ]:
art["text"], art["title"], n_llm = AL.apply_lexicon(art, L)
print(f"replacements from the lexicon: {n_llm.sum():,} in {(n_llm > 0).mean():.1%} of the articles")
art[["article_id", "domain", "title", "date", "text", "score", "label", "level"]].to_csv(C.OUT / "articles_anonymous.csv.gz", index=False)
print("saved", C.OUT / "articles_anonymous.csv.gz", "(the text of every later step)")